# View TRACE synthesis routes

Open a generation CSV to inspect the starting molecule, reaction templates, added reactants and products.
The notebook reads the recorded trajectory and draws each completed step with RDKit.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Leave as None to open the most recent generation under runs/generation/.
ROUTE_CSV = None  # "runs/generation/your_run/rewarded_routes.csv"
TEMPLATE_CSV = None  # Defaults to the template path saved with the run.
ROW_INDEX = 0
FINAL_SMILES = ""  # Set this to look up the first matching route.

if ROUTE_CSV is None:
    candidates = list((ROOT / "runs/generation").glob("*/success_routes.csv"))
    if not candidates:
        raise FileNotFoundError("Set ROUTE_CSV to a generated success_routes.csv or rewarded_routes.csv.")
    csv_path = max(candidates, key=lambda path: path.stat().st_mtime)
else:
    csv_path = ROOT / ROUTE_CSV

routes = pd.read_csv(csv_path, keep_default_na=False)
required = {"Final_Product", "Rxn_Count"}
if not required.issubset(routes.columns):
    raise ValueError(f"Expected {sorted(required)} in {csv_path}")
if routes.empty:
    raise ValueError(f"{csv_path} has no completed routes. Select another run or generate more batches.")

print(f"{len(routes):,} routes in {csv_path}")
summary = ["start_mol", "Start_Reactant", "Final_Product", "Rxn_Count", "Termination", "Fate", "Reward"]
display(routes[[column for column in summary if column in routes.columns]].head())

In [ ]:
if FINAL_SMILES:
    query = Chem.MolFromSmiles(FINAL_SMILES)
    if query is None:
        raise ValueError("FINAL_SMILES is invalid.")
    matches = routes[routes["Final_Product"] == Chem.MolToSmiles(query)]
    if matches.empty:
        raise ValueError("No recorded route matches FINAL_SMILES.")
    row = matches.iloc[0]
else:
    row = routes.iloc[ROW_INDEX]

start_mol = row.get("start_mol") or row.get("Start_Reactant", "")
termination = row.get("Termination", row.get("Fate", ""))
print(f"Start: {start_mol or '(unrecorded)'}")
print(f"Final: {row['Final_Product']}  |  Termination: {termination}")

run_config_path = csv_path.parent / "config.yaml"
run_config = yaml.safe_load(run_config_path.read_text()) if run_config_path.is_file() else {}
template_path = ROOT / (
    TEMPLATE_CSV or run_config.get("template_csv_path", "data/templates/trace_core_templates_top90_update.csv")
)
templates = pd.read_csv(template_path) if template_path.is_file() else None

steps = []
current = start_mol
for step_number in range(1, int(row["Rxn_Count"]) + 1):
    prefix = f"Rxn{step_number}_"
    template_id = int(row[prefix + "Template"])
    reactant = row[prefix + "Reactant"]
    product = row[prefix + "Product"]
    smarts = ""
    if templates is not None:
        smarts = templates.iloc[template_id]["forward_template"]
    steps.append({
        "Step": step_number,
        "Template": template_id,
        "Forward SMARTS": smarts,
        "Input": current,
        "Added reactant": "" if reactant == "N/A" else reactant,
        "Product": product,
    })
    current = product

display(pd.DataFrame(steps))
if current != row["Final_Product"]:
    print("The final product differs from the last recorded product.")

In [ ]:
if start_mol:
    display(Draw.MolToImage(Chem.MolFromSmiles(start_mol), size=(260, 180), legend="Starting molecule"))

for step in steps:
    label = f"Step {step['Step']}  |  Template {step['Template']}"
    if step["Forward SMARTS"]:
        label += f"  |  {step['Forward SMARTS']}"
    print(label)
    labels = ("Input", "Added reactant", "Product")
    molecules = [Chem.MolFromSmiles(step[key]) if step[key] else None for key in labels]
    legends = [label if molecule is not None else "" for label, molecule in zip(labels, molecules)]
    display(Draw.MolsToGridImage(molecules, molsPerRow=3, subImgSize=(240, 180), legends=legends))